# Contaminants

## Imports

In [1]:
import pandas as pd
import pprint as pp
import json

# https://www.datacamp.com/tutorial/settingwithcopywarning-pandas
# https://saturncloud.io/blog/how-to-disable-warnings-in-jupyter-notebook/
import warnings


# Not good, but the warnings were annoying me!!
# warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings(action='ignore')

## Load original contaminants dataset

- Measured air pollutants by the air quality measurement stations of the city of Barcelona

- [Contaminats Dataset](https://opendata-ajuntament.barcelona.cat/data/en/dataset/contaminants-estacions-mesura-qualitat-aire)

In [2]:
bronze_contaminants_csv_file_path: str = \
    './../data/bronze/contaminants/contaminants.csv'

In [3]:
df_bronze_contaminants: pd.DataFrame = \
    pd.read_csv(bronze_contaminants_csv_file_path)

## Original contaminants dataset EDA

In [4]:
# 21 Rows and 3 Columns
df_bronze_contaminants_shape =df_bronze_contaminants.shape

print(f"Bronze Contaminants rows: '{ \
    df_bronze_contaminants_shape[0] }', columns: '{ \
    df_bronze_contaminants_shape[1] }'")

Bronze Contaminants rows: '21', columns: '3'


In [5]:
# Display the first 5 rows of the DataFrame
df_bronze_contaminants.head()

,Codi_Contaminant,Desc_Contaminant,Unitats
0,1,SO2,µg/m³
1,6,CO,mg/m³
2,7,NO,µg/m³
3,8,NO2,µg/m³
4,9,PM2.5,µg/m³


In [6]:
# Display the last 5 rows of the DataFrame
df_bronze_contaminants.tail()

,Codi_Contaminant,Desc_Contaminant,Unitats
16,114,O3*,µg/m³
17,996,Flow 2 (Mesura interna equip),NaN
18,997,Flow 1 (Mesura interna equip),NaN
19,998,Flow C (Mesura interna equip),
20,999,Biomassa Black Carbon,%


In [7]:
# Display all data
df_bronze_contaminants

,Codi_Contaminant,Desc_Contaminant,Unitats
0,1,SO2,µg/m³
1,6,CO,mg/m³
2,7,NO,µg/m³
3,8,NO2,µg/m³
4,9,PM2.5,µg/m³
5,10,PM10,µg/m³
6,12,NOx,µg/m³
7,14,O3,µg/m³
8,22,Black Carbon,µg/m³
9,101,SO2*,µg/m³


In [8]:
# Summary of the DataFrame including column names, their data types, 
# and their non-null values count
df_bronze_contaminants.info()

# 'Codi_Contaminant' -> int code -> Contaminant code
# 'Desc_Contaminant' -> object/string (text) -> Contaminant description
# 'Unitats'          -> object/string (text) -> Contaminant units of measurement

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Codi_Contaminant  21 non-null     int64 
 1   Desc_Contaminant  21 non-null     object
 2   Unitats           19 non-null     object
dtypes: int64(1), object(2)
memory usage: 636.0+ bytes


## Duplicated data and missing values

In [9]:
# Missing values for each column
missing_values = df_bronze_contaminants.isnull().sum()
missing_values

# Result: 2 NULL or NaN values in the 'Unitats' column
# Real result (by observation of dataset): 3 NULL or NaN values in the 'Unitats' column
# Rows 17, 18, 19

Codi_Contaminant    0
Desc_Contaminant    0
Unitats             2
dtype: int64

In [10]:
# Missing values for each column
missing_values = df_bronze_contaminants.isna().sum()
missing_values

# Result: 2 NULL or NaN values in the 'Unitats' column
# Real result (by observation of dataset): 3 NULL or NaN values in the 'Unitats' column
# Rows 17, 18, 19

Codi_Contaminant    0
Desc_Contaminant    0
Unitats             2
dtype: int64

In [11]:
# Get duplicated rows
duplicates = df_bronze_contaminants.duplicated().sum()
print("Duplicates: ", duplicates)

# No duplicated rows
# In reality we have multiple contaminants with 2 ids 
# (same 'Desc_Contaminant' but with an extra '*' char)

Duplicates:  0


In [12]:
# Display all data
df_bronze_contaminants

,Codi_Contaminant,Desc_Contaminant,Unitats
0,1,SO2,µg/m³
1,6,CO,mg/m³
2,7,NO,µg/m³
3,8,NO2,µg/m³
4,9,PM2.5,µg/m³
5,10,PM10,µg/m³
6,12,NOx,µg/m³
7,14,O3,µg/m³
8,22,Black Carbon,µg/m³
9,101,SO2*,µg/m³


## Changing columns names

In [13]:
# Original DataFrame columns names
print(df_bronze_contaminants.columns)

Index(['Codi_Contaminant', 'Desc_Contaminant', 'Unitats'], dtype='object')


In [14]:
# New DataFrame columns names
new_df_columns_names = ['code', 'description', 'unit']

# Change columns of DataFrame
df_bronze_contaminants.columns = new_df_columns_names

In [15]:
# Create a new column 'extended_description' as a copy of 'description'
df_bronze_contaminants['extended_description'] = \
    df_bronze_contaminants['description']

new_df_columns_names.insert(
    new_df_columns_names.index('description') + 1, \
    'extended_description' \
)

In [16]:
# Reorder columns in DataFrame
df_bronze_contaminants = df_bronze_contaminants[new_df_columns_names]

In [17]:
# Display all data
df_bronze_contaminants

,code,description,extended_description,unit
0,1,SO2,SO2,µg/m³
1,6,CO,CO,mg/m³
2,7,NO,NO,µg/m³
3,8,NO2,NO2,µg/m³
4,9,PM2.5,PM2.5,µg/m³
5,10,PM10,PM10,µg/m³
6,12,NOx,NOx,µg/m³
7,14,O3,O3,µg/m³
8,22,Black Carbon,Black Carbon,µg/m³
9,101,SO2*,SO2*,µg/m³


## Remove contaminants 

In [18]:
# Remove contaminants that "don't matter" for the project
contaminants_to_remove_from_df: pd.DataFrame  = [22, 996, 997, 998, 999]

# df_bronze_contaminants is now the old df_bronze_contaminants minus the contaminants
# which the code are in the 'contaminants_to_remove_from_df' list
df_bronze_contaminants = df_bronze_contaminants[
    ~df_bronze_contaminants.code.isin(contaminants_to_remove_from_df)
]

# Display all data
df_bronze_contaminants

,code,description,extended_description,unit
0,1,SO2,SO2,µg/m³
1,6,CO,CO,mg/m³
2,7,NO,NO,µg/m³
3,8,NO2,NO2,µg/m³
4,9,PM2.5,PM2.5,µg/m³
5,10,PM10,PM10,µg/m³
6,12,NOx,NOx,µg/m³
7,14,O3,O3,µg/m³
9,101,SO2*,SO2*,µg/m³
10,106,CO*,CO*,mg/m³


## EDA conclusions

- All contaminants have the __same unit (with the exception of the contaminant 'CO' / 'CO*' using mg/m³)__.
    - __All others__ are measured in __µg/m³__.
- There is a relationship between the __previous contaminant code__ and the __new contaminant code (plus 100)__.
- There is a relationship between the __previous contaminant description__ and the __new contaminant description (added '*' char)__.
- There are 16 entries (contaminants). But since they are 'duplicated', __there are only 8 "valid" contaminants__.

- Contaminant codes marked __with an '*'__ indicate a __higher precision for the data__, which results in __more decimal places__.

- __We will only study/forecast only the 2/3 most important to the city of barcelona__.

In [19]:
df_key_contaminants: pd.DataFrame = df_bronze_contaminants.copy()

## Standardizing text columns

In [20]:
df_key_contaminants.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16 entries, 0 to 16
Data columns (total 4 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   code                  16 non-null     int64 
 1   description           16 non-null     object
 2   extended_description  16 non-null     object
 3   unit                  16 non-null     object
dtypes: int64(1), object(3)
memory usage: 640.0+ bytes


In [21]:
# Standardizing text data

df_key_contaminants.unit = df_key_contaminants.unit.str.strip()
# df_key_contaminants.unit = df_key_contaminants.unit.astype(str)

df_key_contaminants.description = \
    df_key_contaminants.description \
    .str.strip() \
    .str.upper()

# df_key_contaminants.description = df_key_contaminants.description.astype(str)

df_key_contaminants.extended_description = \
    df_key_contaminants.extended_description \
    .str.strip() \
    .str.upper()

# df_key_contaminants.extended_description = \
#   df_key_contaminants.extended_description.astype(str)

In [22]:
# Display all data
df_key_contaminants

,code,description,extended_description,unit
0,1,SO2,SO2,µg/m³
1,6,CO,CO,mg/m³
2,7,NO,NO,µg/m³
3,8,NO2,NO2,µg/m³
4,9,PM2.5,PM2.5,µg/m³
5,10,PM10,PM10,µg/m³
6,12,NOX,NOX,µg/m³
7,14,O3,O3,µg/m³
9,101,SO2*,SO2*,µg/m³
10,106,CO*,CO*,mg/m³


In [23]:
# Remove '*' from the new contaminants description
df_key_contaminants.description = \
    df_key_contaminants.description.str.replace('*', '')

# Replace 'NOX' for 'NOx'
df_key_contaminants.description = \
    df_key_contaminants.description.str.replace('X', 'x')

# Possible problem with data onward (description name can be used as variable names,
# keys or column names)
# PM2.5 -> The '.' can be a problem and will be replaced with '_'
df_key_contaminants.description = \
    df_key_contaminants.description.str.replace('.', '_')

In [24]:
# Remove '*' from the new contaminants extended_description
df_key_contaminants.extended_description = \
    df_key_contaminants.extended_description.str.replace('*', ' (extended)')

# Replace 'NOX' for 'NOx'
df_key_contaminants.extended_description = \
    df_key_contaminants.extended_description.str.replace('X', 'x')

# Possible problem with data onward (description name can be used as variable names,
# keys or column names)
# PM2.5 -> The '.' can be a problem and will be replaced with '_'
df_key_contaminants.extended_description = \
    df_key_contaminants.extended_description.str.replace('.', '_')

In [25]:
df_key_contaminants.loc[ \
    ~df_key_contaminants.extended_description.str.contains('\(extended\)'), \
    'extended_description' \
] += ' (original)'


In [26]:
df_key_contaminants['is_original_contaminant'] = \
    ~df_key_contaminants.extended_description.str.contains('\(extended\)')

In [27]:
df_key_contaminants

,code,description,extended_description,unit,is_original_contaminant
0,1,SO2,SO2 (original),µg/m³,True
1,6,CO,CO (original),mg/m³,True
2,7,NO,NO (original),µg/m³,True
3,8,NO2,NO2 (original),µg/m³,True
4,9,PM2_5,PM2_5 (original),µg/m³,True
5,10,PM10,PM10 (original),µg/m³,True
6,12,NOx,NOx (original),µg/m³,True
7,14,O3,O3 (original),µg/m³,True
9,101,SO2,SO2 (extended),µg/m³,False
10,106,CO,CO (extended),mg/m³,False


In [28]:
# Create a new column 'original_description' as a copy of 'description'
df_key_contaminants['original_description'] = \
    df_key_contaminants.description

# Add '*' to the original contaminants description
df_key_contaminants.loc[ \
    ~df_key_contaminants.is_original_contaminant, \
    'original_description' \
] += '*'

pollutant_code_to_description_map = {
    "SO2": "Sulfur Dioxide",
    "PM2_5": "Fine Particulate Matter",
    "NO2": "Nitrogen Dioxide",
    "NO": "Nitric Oxide",
    "CO": "Carbon Monoxide",
    "PM10": "Coarse Particulate Matter",
    "NOx": "Nitrogen Oxides",
    "O3": "Ozone"
}

df_key_contaminants['text_description'] = \
    df_key_contaminants['description'].map(pollutant_code_to_description_map)

# Reorder columns in DataFrame
df_key_contaminants = df_key_contaminants[['code', 'description', 
    'text_description', 'original_description', 'extended_description',
    'unit', 'is_original_contaminant']]

df_key_contaminants

,code,description,text_description,original_description,extended_description,unit,is_original_contaminant
0,1,SO2,Sulfur Dioxide,SO2,SO2 (original),µg/m³,True
1,6,CO,Carbon Monoxide,CO,CO (original),mg/m³,True
2,7,NO,Nitric Oxide,NO,NO (original),µg/m³,True
3,8,NO2,Nitrogen Dioxide,NO2,NO2 (original),µg/m³,True
4,9,PM2_5,Fine Particulate Matter,PM2_5,PM2_5 (original),µg/m³,True
5,10,PM10,Coarse Particulate Matter,PM10,PM10 (original),µg/m³,True
6,12,NOx,Nitrogen Oxides,NOx,NOx (original),µg/m³,True
7,14,O3,Ozone,O3,O3 (original),µg/m³,True
9,101,SO2,Sulfur Dioxide,SO2*,SO2 (extended),µg/m³,False
10,106,CO,Carbon Monoxide,CO*,CO (extended),mg/m³,False


## Save data to csv file

In [29]:
silver_contaminants_csv_file_path = './../data/silver/contaminants/contaminants.csv'

In [30]:
df_silver_contaminants = df_key_contaminants.copy()
df_silver_contaminants.to_csv(silver_contaminants_csv_file_path, index=False)

In [31]:
df_test_silver_read_csv = pd.read_csv(silver_contaminants_csv_file_path)
df_test_silver_read_csv

,code,description,text_description,original_description,extended_description,unit,is_original_contaminant
0,1,SO2,Sulfur Dioxide,SO2,SO2 (original),µg/m³,True
1,6,CO,Carbon Monoxide,CO,CO (original),mg/m³,True
2,7,NO,Nitric Oxide,NO,NO (original),µg/m³,True
3,8,NO2,Nitrogen Dioxide,NO2,NO2 (original),µg/m³,True
4,9,PM2_5,Fine Particulate Matter,PM2_5,PM2_5 (original),µg/m³,True
5,10,PM10,Coarse Particulate Matter,PM10,PM10 (original),µg/m³,True
6,12,NOx,Nitrogen Oxides,NOx,NOx (original),µg/m³,True
7,14,O3,Ozone,O3,O3 (original),µg/m³,True
8,101,SO2,Sulfur Dioxide,SO2*,SO2 (extended),µg/m³,False
9,106,CO,Carbon Monoxide,CO*,CO (extended),mg/m³,False


## Create and test code to implement ContaminantManager class

In [32]:
# Shortning variable name
df = df_silver_contaminants

In [33]:
# Each contaminant description has a count of 2
df['text_description'].value_counts()

text_description
Sulfur Dioxide               2
Carbon Monoxide              2
Nitric Oxide                 2
Nitrogen Dioxide             2
Fine Particulate Matter      2
Coarse Particulate Matter    2
Nitrogen Oxides              2
Ozone                        2
Name: count, dtype: int64

In [34]:
# Each contaminant has 2 codes
df['description'].value_counts()

description
SO2      2
CO       2
NO       2
NO2      2
PM2_5    2
PM10     2
NOx      2
O3       2
Name: count, dtype: int64

In [35]:
# Get distinct contaminant descriptions
distinct_contaminant_descriptions = set(df.description)

# Dictionary to hold contaminant data without duplications
# and with the multiple contaminants codes in a list 
contaminants_data_dict = {}

In [36]:
for description in distinct_contaminant_descriptions:
    
    # Creates a DataFrame with 2 rows (each contaminants have two codes -> two rows)
    df_unique_contaminant = df[df.description == description]
    
    # Iterate the DataFrame rows
    for index, row in df_unique_contaminant.iterrows():
        
        # Is already a key in the dictionary? No, then is added to it
        if row.description not in contaminants_data_dict:
            
            contaminants_data_dict[row.description] = {}
            contaminants_data_dict[row.description]['unit'] = row.unit
            contaminants_data_dict[row.description]['text_description'] = row.text_description
            contaminants_data_dict[row.description]['codes'] = list()
            contaminants_data_dict[row.description]['original_description'] = dict()
            contaminants_data_dict[row.description]['extended_description'] = dict()
            contaminants_data_dict[row.description]['is_original_contaminant'] = dict()
        
        # Add the code to the list of codes
        contaminants_data_dict[row.description]['codes'].append(row.code)
        contaminants_data_dict[row.description]['original_description'][str(row.code)] = row.original_description
        contaminants_data_dict[row.description]['extended_description'][str(row.code)] = row.extended_description
        contaminants_data_dict[row.description]['is_original_contaminant'][str(row.code)] = row.is_original_contaminant

In [37]:
contaminants_data_dict

{'NO': {'unit': 'µg/m³',
  'text_description': 'Nitric Oxide',
  'codes': [7, 107],
  'original_description': {'7': 'NO', '107': 'NO*'},
  'extended_description': {'7': 'NO (original)', '107': 'NO (extended)'},
  'is_original_contaminant': {'7': True, '107': False}},
 'CO': {'unit': 'mg/m³',
  'text_description': 'Carbon Monoxide',
  'codes': [6, 106],
  'original_description': {'6': 'CO', '106': 'CO*'},
  'extended_description': {'6': 'CO (original)', '106': 'CO (extended)'},
  'is_original_contaminant': {'6': True, '106': False}},
 'O3': {'unit': 'µg/m³',
  'text_description': 'Ozone',
  'codes': [14, 114],
  'original_description': {'14': 'O3', '114': 'O3*'},
  'extended_description': {'14': 'O3 (original)', '114': 'O3 (extended)'},
  'is_original_contaminant': {'14': True, '114': False}},
 'PM10': {'unit': 'µg/m³',
  'text_description': 'Coarse Particulate Matter',
  'codes': [10, 110],
  'original_description': {'10': 'PM10', '110': 'PM10*'},
  'extended_description': {'10': 'PM10

In [38]:
contaminants_data_dict['PM10']

{'unit': 'µg/m³',
 'text_description': 'Coarse Particulate Matter',
 'codes': [10, 110],
 'original_description': {'10': 'PM10', '110': 'PM10*'},
 'extended_description': {'10': 'PM10 (original)', '110': 'PM10 (extended)'},
 'is_original_contaminant': {'10': True, '110': False}}

In [39]:
contaminants_data_dict['CO']

{'unit': 'mg/m³',
 'text_description': 'Carbon Monoxide',
 'codes': [6, 106],
 'original_description': {'6': 'CO', '106': 'CO*'},
 'extended_description': {'6': 'CO (original)', '106': 'CO (extended)'},
 'is_original_contaminant': {'6': True, '106': False}}

In [40]:
contaminants_data_dict['PM2_5']

{'unit': 'µg/m³',
 'text_description': 'Fine Particulate Matter',
 'codes': [9, 109],
 'original_description': {'9': 'PM2_5', '109': 'PM2_5*'},
 'extended_description': {'9': 'PM2_5 (original)', '109': 'PM2_5 (extended)'},
 'is_original_contaminant': {'9': True, '109': False}}

## Save contaminants in JSON format

* [Geeksforgeeks > JSON dump vs JSON dumps](https://www.geeksforgeeks.org/python/python-difference-between-json-dump-and-json-dumps/)
* [DataCamp > JSON tutorial](https://www.datacamp.com/tutorial/json-data-python)

In [41]:
contaminants_json_file_path = './../data/gold/contaminants/contaminants.json'

In [42]:
with open(contaminants_json_file_path, 'w') as json_file:
    json.dump(contaminants_data_dict, json_file)

In [43]:
json_data_test_read = {}

with open(contaminants_json_file_path) as json_file:
    json_data_test_read = json.load(json_file)

In [44]:
json_data_test_read

{'NO': {'unit': 'µg/m³',
  'text_description': 'Nitric Oxide',
  'codes': [7, 107],
  'original_description': {'7': 'NO', '107': 'NO*'},
  'extended_description': {'7': 'NO (original)', '107': 'NO (extended)'},
  'is_original_contaminant': {'7': True, '107': False}},
 'CO': {'unit': 'mg/m³',
  'text_description': 'Carbon Monoxide',
  'codes': [6, 106],
  'original_description': {'6': 'CO', '106': 'CO*'},
  'extended_description': {'6': 'CO (original)', '106': 'CO (extended)'},
  'is_original_contaminant': {'6': True, '106': False}},
 'O3': {'unit': 'µg/m³',
  'text_description': 'Ozone',
  'codes': [14, 114],
  'original_description': {'14': 'O3', '114': 'O3*'},
  'extended_description': {'14': 'O3 (original)', '114': 'O3 (extended)'},
  'is_original_contaminant': {'14': True, '114': False}},
 'PM10': {'unit': 'µg/m³',
  'text_description': 'Coarse Particulate Matter',
  'codes': [10, 110],
  'original_description': {'10': 'PM10', '110': 'PM10*'},
  'extended_description': {'10': 'PM10

## Test ContaminantManagerJSON class

* [Geeksforgeeks import modules from dir](https://www.geeksforgeeks.org/python/python-import-module-from-different-directory/)

In [45]:
from generic_code.ContaminantManagerJSON import ContaminantManagerJSON

In [46]:
contaminant_manager = ContaminantManagerJSON(contaminants_json_file_path)

In [47]:
bar_line = "=" * 80

In [48]:
print(bar_line)
print("Get all contaminant descriptions [ContaminantManagerJSON class]")
print(bar_line)

pp.pprint(contaminant_manager.get_all_contaminants_descriptions())

Get all contaminant descriptions [ContaminantManagerJSON class]
['NO', 'CO', 'O3', 'PM10', 'NO2', 'SO2', 'NOx', 'PM2_5']


In [49]:
print(bar_line)
print("Get all contaminant codes [ContaminantManagerJSON class]")
print(bar_line)

pp.pprint(contaminant_manager.get_all_contaminants_codes())

Get all contaminant codes [ContaminantManagerJSON class]
[1, 101, 6, 7, 8, 9, 106, 107, 10, 108, 14, 110, 12, 112, 114, 109]


In [50]:
print(bar_line)
print("Get contaminant data by description [ContaminantManagerJSON class]")
print(bar_line, '\n')

for description in ("PM10", "PM2_5", "SO2", "CO", "INVALID"):
    data = contaminant_manager.get_contaminant_data_by_description(description)
    print(f"'{description}' data => ")
    pp.pprint(data)
    print('\n')

Get contaminant data by description [ContaminantManagerJSON class]

'PM10' data => 
{'codes': [10, 110],
 'description': 'PM10',
 'extended_description': {'10': 'PM10 (original)', '110': 'PM10 (extended)'},
 'is_original_contaminant': {'10': True, '110': False},
 'original_description': {'10': 'PM10', '110': 'PM10*'},
 'text_description': 'Coarse Particulate Matter',
 'unit': 'µg/m³'}


'PM2_5' data => 
{'codes': [9, 109],
 'description': 'PM2_5',
 'extended_description': {'109': 'PM2_5 (extended)', '9': 'PM2_5 (original)'},
 'is_original_contaminant': {'109': False, '9': True},
 'original_description': {'109': 'PM2_5*', '9': 'PM2_5'},
 'text_description': 'Fine Particulate Matter',
 'unit': 'µg/m³'}


'SO2' data => 
{'codes': [1, 101],
 'description': 'SO2',
 'extended_description': {'1': 'SO2 (original)', '101': 'SO2 (extended)'},
 'is_original_contaminant': {'1': True, '101': False},
 'original_description': {'1': 'SO2', '101': 'SO2*'},
 'text_description': 'Sulfur Dioxide',
 'unit'

In [51]:
print(bar_line)
print("Get contaminant units by description [ContaminantManagerJSON class]")
print(bar_line, '\n')
    
for description in ("PM10", "PM2_5", "SO2", "CO", "INVALID"):
    unit = contaminant_manager.get_units_by_description(description)
    print(f"'{description}' units => {unit}")

Get contaminant units by description [ContaminantManagerJSON class]

'PM10' units => µg/m³
'PM2_5' units => µg/m³
'SO2' units => µg/m³
'CO' units => mg/m³
'INVALID' units => None


In [52]:
print(bar_line)
print("Get contaminant description by code [ContaminantManagerJSON class]")
print(bar_line, '\n')

contaminants_mapping = ((10, "PM10"), (110, "PM10"), (9, "PM2_5"), (109, "PM2_5"),
                         (1, "SO2"), (101, "SO2"), (6, "CO"), (106, "CO"), (999, "INVALID"))

for numeric_code, expected in contaminants_mapping:
    description = contaminant_manager.get_description_by_code(numeric_code)
    print(f"Code '{numeric_code}' (expected: '{expected}') => ")
    pp.pprint(description)
    print('\n')

Get contaminant description by code [ContaminantManagerJSON class]

Code '10' (expected: 'PM10') => 
{'description': 'PM10',
 'extended_description': 'PM10 (original)',
 'is_original_contaminant': True,
 'original_description': 'PM10'}


Code '110' (expected: 'PM10') => 
{'description': 'PM10',
 'extended_description': 'PM10 (extended)',
 'is_original_contaminant': False,
 'original_description': 'PM10*'}


Code '9' (expected: 'PM2_5') => 
{'description': 'PM2_5',
 'extended_description': 'PM2_5 (original)',
 'is_original_contaminant': True,
 'original_description': 'PM2_5'}


Code '109' (expected: 'PM2_5') => 
{'description': 'PM2_5',
 'extended_description': 'PM2_5 (extended)',
 'is_original_contaminant': False,
 'original_description': 'PM2_5*'}


Code '1' (expected: 'SO2') => 
{'description': 'SO2',
 'extended_description': 'SO2 (original)',
 'is_original_contaminant': True,
 'original_description': 'SO2'}


Code '101' (expected: 'SO2') => 
{'description': 'SO2',
 'extended_descrip

In [53]:
print(bar_line)
print("Get codes by description [ContaminantManagerJSON class]")
print(bar_line, '\n')
    
for description in ("PM10", "PM2_5", "SO2", "CO", "INVALID"):
    codes = contaminant_manager.get_codes_by_description(description)
    print(f"get_codes_by_description('{description}') => {codes}")

Get codes by description [ContaminantManagerJSON class]

get_codes_by_description('PM10') => [10, 110]
get_codes_by_description('PM2_5') => [9, 109]
get_codes_by_description('SO2') => [1, 101]
get_codes_by_description('CO') => [6, 106]
get_codes_by_description('INVALID') => None


In [54]:
print(bar_line)
print("Get unit by code [ContaminantManagerJSON class]")
print(bar_line, '\n')
    
for numeric_code in (10, 110, 1, 101, 6, 106, 999):
    unit = contaminant_manager.get_unit_by_code(numeric_code)
    print(f"get_unit_by_code({numeric_code}) => {unit}")

Get unit by code [ContaminantManagerJSON class]

get_unit_by_code(10) => µg/m³
get_unit_by_code(110) => µg/m³
get_unit_by_code(1) => µg/m³
get_unit_by_code(101) => µg/m³
get_unit_by_code(6) => mg/m³
get_unit_by_code(106) => mg/m³
get_unit_by_code(999) => None


In [55]:
print(bar_line)
print("Check contaminant by description [ContaminantManagerJSON class]")
print(bar_line, '\n')
    
for description in ("PM10", "PM2_5", "CO", "INVALID", "  PM10  "):
    exists = contaminant_manager.has_contaminant_by_description(description)
    print(f"has_contaminant_by_description('{description}') => {exists}")

Check contaminant by description [ContaminantManagerJSON class]

has_contaminant_by_description('PM10') => True
has_contaminant_by_description('PM2_5') => True
has_contaminant_by_description('CO') => True
has_contaminant_by_description('INVALID') => False
has_contaminant_by_description('  PM10  ') => True


In [56]:
print(bar_line)
print("Check contaminant by code [ContaminantManagerJSON class]")
print(bar_line, '\n')
    
for numeric_code in (10, 110, 6, 999):
    exists = contaminant_manager.has_contaminant_by_code(numeric_code)
    print(f"has_contaminant_by_code({numeric_code}) => {exists}")

Check contaminant by code [ContaminantManagerJSON class]

has_contaminant_by_code(10) => True
has_contaminant_by_code(110) => True
has_contaminant_by_code(6) => True
has_contaminant_by_code(999) => False


In [57]:
print(bar_line)
print("Check if original contaminant by code [ContaminantManagerJSON class]")
print(bar_line, '\n')
    
for numeric_code in (10, 110, 1, 101, 999):
    is_original = contaminant_manager.is_original_contaminant_by_code(numeric_code)
    print(f"is_original_contaminant_by_code({numeric_code}) => {is_original}")

Check if original contaminant by code [ContaminantManagerJSON class]

is_original_contaminant_by_code(10) => True
is_original_contaminant_by_code(110) => False
is_original_contaminant_by_code(1) => True
is_original_contaminant_by_code(101) => False
is_original_contaminant_by_code(999) => False


In [58]:
print(bar_line)
print("Get contaminant data by code [ContaminantManagerJSON class]")
print(bar_line, '\n')
    
for numeric_code in (10, 110, 1, 101, 999):
    data = contaminant_manager.get_contaminant_data_by_code(numeric_code)
    print(f"get_contaminant_data_by_code({numeric_code}) => ")
    pp.pprint(data)
    print('\n')

Get contaminant data by code [ContaminantManagerJSON class]

get_contaminant_data_by_code(10) => 
{'codes': [10, 110],
 'description': 'PM10',
 'extended_description': {'10': 'PM10 (original)', '110': 'PM10 (extended)'},
 'is_original_contaminant': {'10': True, '110': False},
 'original_description': {'10': 'PM10', '110': 'PM10*'},
 'text_description': 'Coarse Particulate Matter',
 'unit': 'µg/m³'}


get_contaminant_data_by_code(110) => 
{'codes': [10, 110],
 'description': 'PM10',
 'extended_description': {'10': 'PM10 (original)', '110': 'PM10 (extended)'},
 'is_original_contaminant': {'10': True, '110': False},
 'original_description': {'10': 'PM10', '110': 'PM10*'},
 'text_description': 'Coarse Particulate Matter',
 'unit': 'µg/m³'}


get_contaminant_data_by_code(1) => 
{'codes': [1, 101],
 'description': 'SO2',
 'extended_description': {'1': 'SO2 (original)', '101': 'SO2 (extended)'},
 'is_original_contaminant': {'1': True, '101': False},
 'original_description': {'1': 'SO2', '101'

In [59]:
print(bar_line)
print("Get all contaminants data [ContaminantManagerJSON class]")
print(bar_line, '\n')
    
all_data = contaminant_manager.get_all_contaminants_data()
pp.pprint(all_data)

Get all contaminants data [ContaminantManagerJSON class]

{'CO': {'codes': [6, 106],
        'extended_description': {'106': 'CO (extended)', '6': 'CO (original)'},
        'is_original_contaminant': {'106': False, '6': True},
        'original_description': {'106': 'CO*', '6': 'CO'},
        'text_description': 'Carbon Monoxide',
        'unit': 'mg/m³'},
 'NO': {'codes': [7, 107],
        'extended_description': {'107': 'NO (extended)', '7': 'NO (original)'},
        'is_original_contaminant': {'107': False, '7': True},
        'original_description': {'107': 'NO*', '7': 'NO'},
        'text_description': 'Nitric Oxide',
        'unit': 'µg/m³'},
 'NO2': {'codes': [8, 108],
         'extended_description': {'108': 'NO2 (extended)',
                                  '8': 'NO2 (original)'},
         'is_original_contaminant': {'108': False, '8': True},
         'original_description': {'108': 'NO2*', '8': 'NO2'},
         'text_description': 'Nitrogen Dioxide',
         'unit': 'µg/m³'

In [60]:
print(bar_line)
print("Get all contaminants data [ContaminantManagerJSON class]")
print(bar_line, '\n')
    
all_data = contaminant_manager.get_all_contaminants_data()
pp.pprint(all_data)

Get all contaminants data [ContaminantManagerJSON class]

{'CO': {'codes': [6, 106],
        'extended_description': {'106': 'CO (extended)', '6': 'CO (original)'},
        'is_original_contaminant': {'106': False, '6': True},
        'original_description': {'106': 'CO*', '6': 'CO'},
        'text_description': 'Carbon Monoxide',
        'unit': 'mg/m³'},
 'NO': {'codes': [7, 107],
        'extended_description': {'107': 'NO (extended)', '7': 'NO (original)'},
        'is_original_contaminant': {'107': False, '7': True},
        'original_description': {'107': 'NO*', '7': 'NO'},
        'text_description': 'Nitric Oxide',
        'unit': 'µg/m³'},
 'NO2': {'codes': [8, 108],
         'extended_description': {'108': 'NO2 (extended)',
                                  '8': 'NO2 (original)'},
         'is_original_contaminant': {'108': False, '8': True},
         'original_description': {'108': 'NO2*', '8': 'NO2'},
         'text_description': 'Nitrogen Dioxide',
         'unit': 'µg/m³'

In [61]:
print(bar_line)
print("ContaminantManagerJSON.to_dataframe() output")
print(bar_line, '\n')
    
df = contaminant_manager.to_dataframe()
df

ContaminantManagerJSON.to_dataframe() output



,code,description,text_description,original_description,extended_description,unit,is_original_contaminant
0,7,NO,Nitric Oxide,NO,NO (original),µg/m³,True
1,107,NO,Nitric Oxide,NO*,NO (extended),µg/m³,False
2,6,CO,Carbon Monoxide,CO,CO (original),mg/m³,True
3,106,CO,Carbon Monoxide,CO*,CO (extended),mg/m³,False
4,14,O3,Ozone,O3,O3 (original),µg/m³,True
5,114,O3,Ozone,O3*,O3 (extended),µg/m³,False
6,10,PM10,Coarse Particulate Matter,PM10,PM10 (original),µg/m³,True
7,110,PM10,Coarse Particulate Matter,PM10*,PM10 (extended),µg/m³,False
8,8,NO2,Nitrogen Dioxide,NO2,NO2 (original),µg/m³,True
9,108,NO2,Nitrogen Dioxide,NO2*,NO2 (extended),µg/m³,False


In [62]:
print(bar_line)
print("Get contaminant text description by code [ContaminantManagerJSON class]")
print(bar_line, '\n')
        
for numeric_code in (10, 110, 1, 101, 6, 106, 999):
    text_description = contaminant_manager.get_text_description_by_code(numeric_code)
    print(f"get_text_description_by_code({numeric_code}) => {text_description}")


print('\n')
print(bar_line)
print("Get contaminant text description by description [ContaminantManagerJSON class]")
print(bar_line, '\n')
        
for description in ("PM10", "PM2_5", "SO2", "CO", "INVALID"):
    text_description = contaminant_manager.get_text_description_by_description(description)
    print(f"'{description}' text description => '{text_description}'")

Get contaminant text description by code [ContaminantManagerJSON class]

get_text_description_by_code(10) => Coarse Particulate Matter
get_text_description_by_code(110) => Coarse Particulate Matter
get_text_description_by_code(1) => Sulfur Dioxide
get_text_description_by_code(101) => Sulfur Dioxide
get_text_description_by_code(6) => Carbon Monoxide
get_text_description_by_code(106) => Carbon Monoxide
get_text_description_by_code(999) => None


Get contaminant text description by description [ContaminantManagerJSON class]

'PM10' text description => 'Coarse Particulate Matter'
'PM2_5' text description => 'Fine Particulate Matter'
'SO2' text description => 'Sulfur Dioxide'
'CO' text description => 'Carbon Monoxide'
'INVALID' text description => 'None'


In [63]:
%run generic_code/ContaminantManagerJSON.py

Current directory: c:\Users\RafaelBranco\Desktop\2526_air_quality_forecast_msc_project\code 

Get all contaminant descriptions [ContaminantManagerJSON class]
['NO', 'CO', 'O3', 'PM10', 'NO2', 'SO2', 'NOx', 'PM2_5']


Get all contaminant codes [ContaminantManagerJSON class]
[1, 101, 6, 7, 8, 9, 106, 107, 10, 108, 14, 110, 12, 112, 114, 109]


Get contaminant data by description [ContaminantManagerJSON class]

'PM10' data => 
{'codes': [10, 110],
 'description': 'PM10',
 'extended_description': {'10': 'PM10 (original)', '110': 'PM10 (extended)'},
 'is_original_contaminant': {'10': True, '110': False},
 'original_description': {'10': 'PM10', '110': 'PM10*'},
 'text_description': 'Coarse Particulate Matter',
 'unit': 'µg/m³'}


'PM2_5' data => 
{'codes': [9, 109],
 'description': 'PM2_5',
 'extended_description': {'109': 'PM2_5 (extended)', '9': 'PM2_5 (original)'},
 'is_original_contaminant': {'109': False, '9': True},
 'original_description': {'109': 'PM2_5*', '9': 'PM2_5'},
 'text_descr